In [15]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01/bank-additional-full.csv
/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01/bank-additional-names.txt
/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01/bank-additional.csv
/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01/bank-full.csv
/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01/bank-names.txt
/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01/bank.csv


In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path(
    "/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01"
)

In [17]:
df3 = pd.read_csv(DATA_DIR / "bank-additional-full.csv", sep=";")
df3.drop(columns=['duration'], inplace=True)
#dropped to prevent data leakage
df3.shape

(41188, 20)

In [18]:
#splitting the dataset into X and y and train test and validation splits
y = df3['y'].map({'no': 0, 'yes': 1})
X=df3.drop(columns=['y'])

# train test split

In [19]:
from sklearn.model_selection import train_test_split

# 70% train, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# Split remaining 30% equally: 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)
for name, target in [
    ("Train", y_train),
    ("Validation", y_val),
    ("Test", y_test)
]:
    print(f"\n{name}")
    print(target.value_counts(normalize=True).round(3))


Train
y
0    0.887
1    0.113
Name: proportion, dtype: float64

Validation
y
0    0.887
1    0.113
Name: proportion, dtype: float64

Test
y
0    0.887
1    0.113
Name: proportion, dtype: float64


stratified split preserves target variable distribution across train test and split

In [20]:
from pathlib import Path

output_dir = Path("/kaggle/working/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

X_train.to_csv(output_dir / "X_train.csv", index=False)
X_val.to_csv(output_dir / "X_val.csv", index=False)
X_test.to_csv(output_dir / "X_test.csv", index=False)

y_train.to_csv(output_dir / "y_train.csv", index=False)
y_val.to_csv(output_dir / "y_val.csv", index=False)
y_test.to_csv(output_dir / "y_test.csv", index=False)

In [21]:
def calculate_woe_iv(X, y, max_bins=10):
    
    data = X.copy()
    data["target"] = np.array(y)
    
    iv_summary = []
    woe_tables = []
    
    for feature in X.columns:
        
        temp = data[[feature, "target"]].copy()
        
        # --------------------------------------------------
        # 1. Create bins for numerical variables
        # --------------------------------------------------
        if pd.api.types.is_numeric_dtype(temp[feature]):
            
            try:
                temp["bin"] = pd.qcut(
                    temp[feature],
                    q=max_bins,
                    duplicates="drop"
                )
            except:
                temp["bin"] = temp[feature].astype(str)
        
        # --------------------------------------------------
        # 2. Keep categories as they are
        # --------------------------------------------------
        else:
            temp["bin"] = temp[feature].fillna("Missing")
        
        # --------------------------------------------------
        # 3. Calculate counts
        # --------------------------------------------------
        summary = (
            temp.groupby("bin", observed=False)["target"]
            .agg(
                Total="size",
                Events="sum"
            )
            .reset_index()
        )
        
        summary["Non-Events"] = (
            summary["Total"] - summary["Events"]
        )
        
        # --------------------------------------------------
        # 4. Distribution of events/non-events
        # --------------------------------------------------
        total_events = summary["Events"].sum()
        total_non_events = summary["Non-Events"].sum()
        
        # Small smoothing to avoid division by zero
        n_bins = len(summary)
        
        summary["Event Distribution"] = (
            (summary["Events"] + 0.5) /
            (total_events + 0.5 * n_bins)
        )
        
        summary["Non-Event Distribution"] = (
            (summary["Non-Events"] + 0.5) /
            (total_non_events + 0.5 * n_bins)
        )
        
        # --------------------------------------------------
        # 5. Weight of Evidence
        # Convention:
        # WoE = ln(Non-event distribution / Event distribution)
        # --------------------------------------------------
        summary["WoE"] = np.log(
            summary["Non-Event Distribution"] /
            summary["Event Distribution"]
        )
        
        # --------------------------------------------------
        # 6. Information Value
        # --------------------------------------------------
        summary["IV"] = (
            summary["Non-Event Distribution"] -
            summary["Event Distribution"]
        ) * summary["WoE"]
        
        feature_iv = summary["IV"].sum()
        
        # --------------------------------------------------
        # 7. Store IV summary
        # --------------------------------------------------
        iv_summary.append({
            "Feature": feature,
            "IV": feature_iv
        })
        
        # --------------------------------------------------
        # 8. Store detailed WoE table
        # --------------------------------------------------
        summary.insert(0, "Feature", feature)
        
        woe_tables.append(summary)
    
    iv_report = (
        pd.DataFrame(iv_summary)
        .sort_values("IV", ascending=False)
        .reset_index(drop=True)
    )
    
    woe_report = pd.concat(
        woe_tables,
        ignore_index=True
    )
    
    return iv_report, woe_report

In [22]:
iv_report, woe_report = calculate_woe_iv(
    X_train,
    y_train,
    max_bins=10
)
iv_report.round(4)

,Feature,IV
0,nr.employed,1.1363
1,euribor3m,1.0123
2,cons.conf.idx,0.8341
3,emp.var.rate,0.7715
4,poutcome,0.5474
5,month,0.4728
6,cons.price.idx,0.4136
7,contact,0.2431
8,previous,0.2172
9,job,0.1772


In [23]:
for col in X_train.columns:
    print(f"\n{'='*50}")
    print(f"WoE for: {col}")
    print(f"{'='*50}")
    
    print(
        woe_report[
            woe_report["Feature"] == col
        ].round(4)
    )


WoE for: age
  Feature             bin  Total  Events  Non-Events  Event Distribution  \
0     age  (16.999, 28.0]   2981     521        2460              0.1603   
1     age    (28.0, 31.0]   3576     437        3139              0.1345   
2     age    (31.0, 33.0]   2585     265        2320              0.0816   
3     age    (33.0, 35.0]   2449     247        2202              0.0761   
4     age    (35.0, 38.0]   3194     293        2901              0.0902   
5     age    (38.0, 41.0]   2737     217        2520              0.0669   
6     age    (41.0, 45.0]   2999     234        2765              0.0721   
7     age    (45.0, 49.0]   2660     213        2447              0.0656   
8     age    (49.0, 55.0]   3157     310        2847              0.0955   
9     age    (55.0, 98.0]   2493     511        1982              0.1572   

   Non-Event Distribution     WoE      IV  
0                  0.0962 -0.5111  0.0328  
1                  0.1227 -0.0918  0.0011  
2                

In [24]:
print('nr.employed', df3['nr.employed'].nunique())
print('emp.var.rate', df3['emp.var.rate'].nunique())
print('euribor3m', df3['euribor3m'].nunique())
print('cons.conf.idx', df3['cons.conf.idx'].nunique())
print("poutcome", df3['poutcome'].nunique())

nr.employed 11
emp.var.rate 10
euribor3m 316
cons.conf.idx 26
poutcome 3


In [25]:
# for b in [5, 10, 20]:
#     result = calculate_iv(
#         X_train[['euribor3m']],
#         y_train,
#         bins=b
#     )

#     print(b, result)

Information value shoots up: 1) actual predictive value 2) leakage [is the feature available at the time of decision making] 3) binning strategy [more bins more event and non event distiction] 

# Encoding


In [26]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Identify numerical and categorical columns
cat_cols = X_train.select_dtypes(
    include=['object', 'category']
).columns.tolist()

num_cols = X_train.select_dtypes(
    include=np.number
).columns.tolist()

# Numerical preprocessing
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

# Categorical preprocessing
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore',
        drop='first'
    ))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# Fit on train; transform validation and test
X_train_encoded = preprocessor.fit_transform(X_train)
X_val_encoded = preprocessor.transform(X_val)
X_test_encoded = preprocessor.transform(X_test)

print("Train:", X_train_encoded.shape)
print("Validation:", X_val_encoded.shape)
print("Test:", X_test_encoded.shape)

Train: (28831, 52)
Validation: (6178, 52)
Test: (6179, 52)
